In [1]:
import os
from pyspark import SparkConf, SparkContext
from pyspark.sql import SparkSession

os.environ["SPARK_LOCAL_IP"] = "localhost"

conf = (
    SparkConf()
    .setMaster("local[*]")
    .setAppName("Delta Lake Spark Jupyter Lab")
    .set("spark.driver.memory", "30g")
    .set("spark.executor.memory", "30g")
    .set("spark.driver.cores", "16")
    .set("spark.worker.cores", "16")
    .set("spark.jars", "/home/baptvit/Documents/github/mineracao-dados-massivos/apps/jars/*")
    .set("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .set(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
    .set(
        "spark.sql.delta.catalogImpl", "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )
)
sc = SparkContext(conf=conf).getOrCreate()
spark: SparkSession = SparkSession(sc)

24/06/29 11:03:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/06/29 11:03:18 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
24/06/29 11:03:18 ERROR SparkContext: Failed to add /home/baptvit/Documents/github/mineracao-dados-massivos/apps/jars/* to Spark environment
java.io.FileNotFoundException: Jar /home/baptvit/Documents/github/mineracao-dados-massivos/apps/jars/* not found
	at org.apache.spark.SparkContext.addLocalJarFile$1(SparkContext.scala:2100)
	at org.apache.spark.SparkContext.addJar(SparkContext.scala:2156)
	at org.apache.spark.SparkContext.$anonfun$new$15(SparkContext.scala:526)
	at org.apache.spark.SparkContext.$anonfun$new$15$adapted(SparkContext.scala:526)
	at scala.collection.mutable.ResizableArray.foreach(ResizableArray.scala:62

In [2]:
spark

In [2]:
import pandas as pd

In [5]:
DF_KB = pd.read_parquet("/home/baptvit/Documents/github/mineracao-dados-massivos/data/med-qa-dataset/medqa_kb")

In [6]:
DF_KB.count()

sentence              861432
embedding_sentence    861432
token_sentence        861432
metadata              861432
dtype: int64

In [7]:
DF_KB.head()

,sentence,embedding_sentence,token_sentence,metadata
0,HCC主要由肝动脉供血，90％以上肿瘤血供丰富；肿瘤压迫周围肝组织可形成假包膜；HCC容易侵...,"[-0.04214775934815407, 0.5282214879989624, 0.4...",127,"Raw file name is: 医学影像学.txt, Were read at: 202..."
1,这种二级－结构是阻止转录继续向下游推进的关键。,"[-0.05957504361867905, 0.22949005663394928, 0....",25,"Raw file name is: all_books.txt, Were read at:..."
2,The holosystolic murmur of chronic TR is gener...,"[-0.20869624614715576, -0.09695304185152054, 0...",98,"Raw file name is: InternalMed_Harrison.txt, We..."
3,Various systemic agents can be used for severe...,"[-0.3980881869792938, 0.006496873218566179, 0....",115,"Raw file name is: InternalMed_Harrison.txt, We..."
4,积极控制高血压是预防心力衰竭发生和防止心力衰竭进展的重要措施。,"[-0.08473484963178635, 0.39388588070869446, -0...",33,"Raw file name is: 临床药理学.txt, Were read at: 202..."


In [9]:
DF_KB.dtypes

sentence              object
embedding_sentence    object
token_sentence         int32
metadata              object
dtype: object

In [12]:
DF_KB[["embedding_sentence"]]

,embedding_sentence
0,"[-0.04214775934815407, 0.5282214879989624, 0.4..."
1,"[-0.05957504361867905, 0.22949005663394928, 0...."
2,"[-0.20869624614715576, -0.09695304185152054, 0..."
3,"[-0.3980881869792938, 0.006496873218566179, 0...."
4,"[-0.08473484963178635, 0.39388588070869446, -0..."
...,...
861427,"[-0.2774333953857422, 0.012855660170316696, 0...."
861428,"[-0.35646873712539673, -0.06070423126220703, 0..."
861429,"[-0.34643250703811646, -0.01157051045447588, 0..."
861430,"[-0.32183653116226196, 0.013590732589364052, 0..."


In [11]:
import numpy as np 

DF_KB['embedding_sentence'] = DF_KB['embedding_sentence'].apply(eval).apply(np.array)

SyntaxError: source code string cannot contain null bytes (<string>)

# Evaluation para Textbook

In [13]:
import pandas as pd

pd_textbook_test = pd.read_parquet("/home/baptvit/Documents/github/mineracao-dados-massivos/evaluation/evaluation/data/pd_textbook_test_embedding_1000")

In [14]:
pd_textbook_test.count()

Unnamed: 0                       1000
sentence                         1000
embedding_sentence               1000
token_sentence                   1000
metadata                         1000
rewrited_sentence                1000
rewrited_sentence_embedding      1000
rewrited_sentence_token_count    1000
dtype: int64

## Pandas Euclidian DIstance

In [20]:
from scipy.spatial import distance as distance_euclidean

def evaluate_top10_euclidean(rewrited_sentence_embedding_value, df=DF_KB):
    def distance_udf(embedding_sentence, rewrited_sentence_embedding):
        return float(distance_euclidean.euclidean(rewrited_sentence_embedding, embedding_sentence))
    df['distances'] = df.apply(lambda row: distance_udf(row['embedding_sentence'], rewrited_sentence_embedding_value), axis=1)
    return df.sort_values(by=['distances']).head(10)

## Euclidian distance

In [9]:
from scipy.spatial import distance as distance_euclidean

In [10]:
from pyspark.sql.types import ArrayType, DoubleType, FloatType
import pyspark.sql.functions as F
from scipy.spatial import distance as distance_euclidean

def evaluate_top10_euclidean(pd_row):
    rewrited_sentence_embedding = pd_row["rewrited_sentence_embedding"]
    distance_udf = F.udf(lambda x: float(distance_euclidean.euclidean(rewrited_sentence_embedding, x)), FloatType())
    distance_df = DF_KB.withColumn('distances', distance_udf(F.col('embedding_sentence'))).sort(F.col("distances").asc()).limit(10)
    return distance_df

# Display the result
#distance_df.show()

In [27]:
pd_textbook_test_1 = pd_textbook_test

In [ ]:
import datetime
import time

algo_name = "euclidean_distance" 
algoritimos_name = []
times = []
timestamps = []
top_1_result = []  # If true, the model is write on top 1 else false
top_5_result = (
    []
)  # If true, the model is write on top 5 results else false
top_10_result = (
    []
)  # If true, the model is write on top 10 results else false
distances = []
query_id_result = []

for rows in pd_textbook_test_1.iterrows():
    algoritimos_name.append(algo_name)
    row = rows[1] 
    query_id = row[0]
    expected_output = row["sentence"]
    rewrited_sentence_embedding = row["rewrited_sentence_embedding"]

    timestamp = datetime.datetime.now()
    timestamps.append(timestamp)
    start_time = time.time()
    #df_top10 = evaluate_top10_euclidean(row)
    df_top10 = evaluate_top10_euclidean(rewrited_sentence_embedding)
    time_query = time.time() - start_time
    times.append(time_query)
    
    #for num, row in enumerate(df_top10):
    for num, rows in enumerate(df_top10.iterrows()):
        row_top_10 = rows[1] 
        # If is first hit
        top_1 = False
        top_5 = False
        top_10 = False
        retrived_sentence = row_top_10["sentence"]
        retrived_distance = row_top_10["distances"]
        if retrived_sentence == expected_output and num == 0:
            top_1 = True
            top_5 = True
            top_10 = True
            distance = retrived_distance
            break
        elif retrived_sentence == expected_output and num <= 5:
            top_1 = False
            top_5 = True
            top_10 = True
            distance = retrived_distance
            break
        elif retrived_sentence == expected_output and num <= 10:
            top_1 = False
            top_5 = False
            top_10 = True
            distance = retrived_distance
            break
        else:
            top_1 = False
            top_5 = False
            top_10 = False
            distance = 1

    distances.append(distance)

    top_1_result.append(top_1)
    top_5_result.append(top_5)
    top_10_result.append(top_10)
    query_id_result.append(query_id)

    msg = f"Query id {query_id},Top 1: {top_1}, Within Top 5 {top_5}, Within Top 10 {top_10} Query time: {time_query}s, Distance {retrived_distance} and with"
    print(msg)

    metrics = {
            "timestamp": [timestamp],
            "algoritimos_name": [algo_name],
            "query_id": [query_id],
            "search_duration": [time_query],
            "top1": [top_1],
            "top5": [top_5],
            "top10": [top_10],
            "distance": [distance],
        }
    df = pd.DataFrame(metrics)
    df.to_csv(f"/home/baptvit/Documents/github/mineracao-dados-massivos/evaluation/evaluation/results/{algo_name}_incremental.csv", mode='a', header=False, index=False)


metrics = {
        "timestamp": timestamps,
        "algoritimos_name": algoritimos_name,
        "query_id": query_id_result,
        "search_duration": times,
        "top1": top_1_result,
        "top5": top_5_result,
        "top10": top_10_result,
        "distance": distances,
    }
df = pd.DataFrame(metrics)
df.to_csv(f"/home/baptvit/Documents/github/mineracao-dados-massivos/evaluation/evaluation/results/{algo_name}_bulk.csv")

/tmp/ipykernel_203391/2306816081.py:21: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  query_id = row[0]


Query id 0,Top 1: True, Within Top 5 True, Within Top 10 True Query time: 6.057042360305786s, Distance 2.5761719852553 and with
Query id 2,Top 1: True, Within Top 5 True, Within Top 10 True Query time: 6.179703950881958s, Distance 1.2964183452624707 and with
Query id 3,Top 1: True, Within Top 5 True, Within Top 10 True Query time: 5.95591926574707s, Distance 1.8630950489794227 and with
Query id 4,Top 1: True, Within Top 5 True, Within Top 10 True Query time: 6.031469821929932s, Distance 1.8714015411267966 and with
Query id 5,Top 1: True, Within Top 5 True, Within Top 10 True Query time: 6.071065187454224s, Distance 1.2275847935409028 and with
Query id 7,Top 1: True, Within Top 5 True, Within Top 10 True Query time: 6.063711643218994s, Distance 2.0747395983615156 and with
Query id 8,Top 1: True, Within Top 5 True, Within Top 10 True Query time: 6.062694311141968s, Distance 1.2752392426026236 and with
Query id 9,Top 1: True, Within Top 5 True, Within Top 10 True Query time: 5.96085000038